# 82 — Selectivity-Aware Prediction

**Key insight:** The counter-assay (PXR-null) measures cytotoxicity, not PXR activation.
- High pEC50(PXR) + high pEC50(null) → cytotoxic, not selective
- High pEC50(PXR) + low pEC50(null) → true PXR agonist (selective)

Selectivity = pEC50(PXR) − pEC50(null) captures the *signal* we care about.

Strategy:
1. Train a selectivity model on the 2,858 compounds with both measurements
2. Train a null-predictor on counter-assay data (for compounds without null measurements)
3. Final: pEC50_pred = selectivity_pred + null_pred

This decomposes the problem into: "is this compound active?" (selectivity)
and "how potent is it non-selectively?" (null). Cleaner signal for both models.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and len(cp)>0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from pxr.data import load_counter
from pxr.chem import to_inchikey

ctr = load_counter().dropna(subset=["smiles","pec50"]).rename(columns={"pec50":"pec50_null"})
ctr["ik"] = ctr["smiles"].map(to_inchikey)
# Deduplicate by InChIKey to prevent row explosion in left-merge
ctr_dedup = ctr[["ik","pec50_null"]].dropna(subset=["ik"]).groupby("ik", as_index=False)["pec50_null"].mean()
tr["ik"]  = tr["smiles"].map(to_inchikey)

# Merge: compounds with both CRC and counter-assay measurements
tr_merged = tr.merge(ctr_dedup, on="ik", how="left")
assert len(tr_merged) == len(tr), f"Merge expanded rows: {len(tr_merged)} vs {len(tr)}"
has_null = tr_merged["pec50_null"].notna().values
selectivity = (tr_merged["pec50"].values - tr_merged["pec50_null"].fillna(0).values).astype(np.float32)
print(f"Compounds with both CRC+null: {has_null.sum()} / {len(tr)}")
print(f"Selectivity distribution (where available):")
print(pd.Series(selectivity[has_null]).describe().round(3).to_string())

Compounds with both CRC+null: 2647 / 4139
Selectivity distribution (where available):
count    2647.000
mean        1.607
std         1.349
min        -4.370
25%         0.545
50%         1.720
75%         2.720
max         4.765


In [5]:
# Model 1: Selectivity predictor (pEC50 - pEC50_null)
print("\n=== Model 1: Selectivity (pEC50 - pEC50_null) ===", flush=True)
sel_target = selectivity.copy()
sel_target[~has_null] = np.nan  # only use compounds with both measurements

oof_sel = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    tr_f = tr_idx[has_null[tr_idx]]
    va_f = va_idx[has_null[va_idx]]
    if len(tr_f) < 50 or len(va_f) < 10: continue
    m = lgb.train(LGBM, lgb.Dataset(X_tr[tr_f], label=sel_target[tr_f]),
                  valid_sets=[lgb.Dataset(X_tr[va_f], label=sel_target[va_f])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_sel[va_idx] = m.predict(X_tr[va_idx])
valid = has_null & np.isfinite(oof_sel)
print(f"Selectivity OOF RAE (where available): {rae(sel_target[valid], oof_sel[valid]):.4f}")

# Model 2: pEC50_null predictor
print("\n=== Model 2: pEC50_null predictor ===", flush=True)
null_target = tr_merged["pec50_null"].values.astype(np.float32)
oof_null = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    tr_f = tr_idx[has_null[tr_idx]]
    va_f = va_idx[has_null[va_idx]]
    if len(tr_f) < 50 or len(va_f) < 10: continue
    m = lgb.train(LGBM, lgb.Dataset(X_tr[tr_f], label=null_target[tr_f]),
                  valid_sets=[lgb.Dataset(X_tr[va_f], label=null_target[va_f])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_null[va_idx] = m.predict(X_tr[va_idx])



=== Model 1: Selectivity (pEC50 - pEC50_null) ===


Selectivity OOF RAE (where available): 0.8777

=== Model 2: pEC50_null predictor ===


In [6]:
# Reconstruct pEC50 = selectivity + null
oof_reconstructed = oof_sel + oof_null
valid_both = np.isfinite(oof_reconstructed)
print(f"Reconstructed pEC50 (selectivity+null): {valid_both.sum()} valid")

# Compare with direct model
oof_direct = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m.predict(X_tr[va_idx])

# Blend: where we have reconstructed predictions, blend; elsewhere use direct
oof = oof_direct.copy()
oof[valid_both] = 0.4 * oof_reconstructed[valid_both] + 0.6 * oof_direct[valid_both]

m_dir = full_metrics(y_tr, oof_direct, cliff_pairs, "direct")
m_rec = full_metrics(y_tr[valid_both], oof_reconstructed[valid_both],
                     label="sel+null_reconstructed")
m_blnd= full_metrics(y_tr, oof, cliff_pairs, "blended")
print("\n" + pd.DataFrame([m_dir, m_rec, m_blnd],
      index=["direct","reconstructed","blended"]).round(4).to_string())


Reconstructed pEC50 (selectivity+null): 4139 valid


  [direct] RAE=0.5643 MAE=0.5134 R²=0.5991 r=0.7740 ρ=0.7268 τ=0.5345  Cliff=nan
  [sel+null_reconstructed] RAE=0.7330 MAE=0.6669 R²=0.3516 r=0.6285 ρ=0.5704 τ=0.4002
  [blended] RAE=0.5917 MAE=0.5384 R²=0.5660 r=0.7632 ρ=0.7211 τ=0.5275  Cliff=nan

                  RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
direct         0.5643  0.5134  0.5991   0.7740    0.7268   0.5345        NaN
reconstructed  0.7330  0.6669  0.3516   0.6285    0.5704   0.4002        NaN
blended        0.5917  0.5384  0.5660   0.7632    0.7211   0.5275        NaN


In [7]:
# Final models
m_sel_final = lgb.train(LGBM, lgb.Dataset(X_tr[has_null], label=sel_target[has_null]),
                         callbacks=[lgb.log_evaluation(-1)])
m_null_final = lgb.train(LGBM, lgb.Dataset(X_tr[has_null], label=null_target[has_null]),
                          callbacks=[lgb.log_evaluation(-1)])
m_direct_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])

te_sel  = m_sel_final.predict(X_te)
te_null = m_null_final.predict(X_te)
te_rec  = te_sel + te_null
te_dir  = m_direct_final.predict(X_te)
te_preds = np.clip(0.4*te_rec + 0.6*te_dir, y_tr.min()-0.5, y_tr.max()+0.5)
print(f"Selectivity preds: mean={te_sel.mean():.2f}  null preds: mean={te_null.mean():.2f}")
np.save(DATA_PROCESSED/"oof_selectivity_aware.npy", oof)
np.save(DATA_PROCESSED/"te_oof_selectivity_aware.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"82_selectivity_aware.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Selectivity preds: mean=1.89  null preds: mean=3.05
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\82_selectivity_aware.csv
Test: min=2.74 med=4.96 max=6.05
